# Dataset Regeneration

### Columnas DataSet Original

- `customer_id`: Identificador único del cliente (C000123)
- `edad`: Edad del cliente en años (35)
- `segmento`: Segmento comercial del cliente (Mass, Select, Premier)
- `antiguedad_meses`: Tiempo como cliente del banco (48)
- `ingreso_mensual`: Ingreso mensual estimado (MXN) (45000)
- `tarjeta_santander`: Producto de tarjeta de crédito contratado (LikeU, Free, Platinum)
- `uso_tarjeta_pct`: Porcentaje estimado de uso de la tarjeta (0.75)
- `transacciones_mes`: Número promedio de compras al mes (18)
- `gasto_mensual`: Monto promedio gastado con la tarjeta al mes (MXN) (12500)
- `ecommerce_pct`: Porcentaje de compras realizadas en línea (0.4)
- `pagos_puntuales_pct`: Porcentaje histórico de pagos realizados a tiempo (0.98)
- `app_logins_mes`: Número de accesos a la App Santander por mes (12)
- `nps`: Net Promoter Score individual (0-10) (8)
- `satisfaccion`: Nivel de satisfacción percibida (1-5) (4.2)
- `abandono_riesgo`: Probabilidad estimada de abandono (0-1) (0.25)
- `activacion_promos_pct`: Porcentaje de promociones u ofertas activadas por el cliente (0.35)

## Customer ID (`customer_id`)

Del formato `C######` (e.g. `C000001`) asignados en orden.

## Edad (`edad`)

- 15-24: 22.244.425 (21.9115815%)
- 25-34: 21.105.726 (20.7899209%)
- 35-44: 18.729.814 (18.4495597%)
- 45-54: 15.640.829 (15.4067952%)
- 55-64: 12.087.127 (11.9062672%)
- 65-74: 7.384.543 (07.274048%)
- 75+: 4.326.566 (04.2618276%)

Total: 101.519.030

Las edades especificas son muestreadas uniformemente dentro de cada rango.

https://www.populationpyramid.net/es/m%C3%A9xico/2026/

## Antiguedad Meses (`transacciones_mes`)

La antiguedad en meses es muestreada de forma uniforme entre 1 mes y (edad - 15) * 12 + 1

## Ingreso Mensual (`ingreso_mensual`)

El ingreso mensual estaría seccionado a partir de las edades. El ingreso es una distribucion log-normal con media con la siguiente data:

https://www.economia.gob.mx/datamexico/es/profile/occupation/profesionistas-y-tecnicos?quarters2=20261&typeJob3=totalOption

La desivación estandar es un supuesto propio definido como: 500 (MXN)

## Gasto Mensual (`gasto_mensual`)

El gasto mensual estaría seccionado a partir de las edades. El gasto sigue de la distribucion log-normal del ingreso mensual con la media desplaza por (0.1*media ingreso normal). Se ajusta un techo para que el gasto mensual no sea mayor por un 10% que el ingreso mensual.

La desivación estandar es un supuesto propio definido como: 500 (MXN)

Nota: El gasto mensual sólo es el generado por la tarjeta de credito. Por lo tanto, escrictamente hablando, es posible que en un mes puntual se gaste más por tarjeta de crédito que los ingresos. Sin embargo, esto reduciría el límite disponible de crédito para el siguiente mes, obligando al cliente a pagar para seguir utilizando la tarjeta. En promedio, este caso se reduce a los meses. Esta variación se ve reflejada entre ambas distribuciones log-normales. 

## Pagos Puntuales (%) (`pagos_puntuales_pct`)

A partir de la diferencia entre ingreso mensual y gasto mensual, hay una correlación positiva donde a mayor restante, mayor es porcentaje de pagos puntuales. Los pagos puntales tienen un rango entre [0, 100].

Formula: `clamp(0,  ingreso mensual - gasto mensual / ingreso mensual * 0.25 + ruido, 100)`

## Uso Ecommerce (`ecommerce_pct`)

El uso ecommerce es directamente proporcional al gasto mensual e inversamente proporcional a la edad. Su rango es de [0, 100]

Formula: `clamp(0,  70 * gasto_mensual / 20 000 - 30*(100 - edad) + ruido, 100)`

## Transacciones Mes

Las transacciones al mes son directamente proporcional al gasto mensual

Formula: `gasto mensual / 200 + ruido`

## Logins al Mes

Los logisn al mes son directamente proprocionales al uso de e-commerce y a las transacciones al mes:

Formula: `clamp(0,  70 * gasto_mensual / 20 000 - 30*(100 - edad) + ruido, 100)`

## NPS

Distribución normal 

Promedio de NPS: 6.92 

https://www.santander.com.mx/personas/acerca-del-banco/responsabilidad-social/assets/pdf/informe-anual/resumen-ejecutivo-del-informe-anual-integrado-2025-v1.pdf

La desviación estandar es un supuesto propio definido como 1.5.

## Satisfaccion	

Distribución normal 

El promedio de satisfacción que se tiene registrado es de 80.1%. En este caso, la satisfacción tiene un rango de [0, 5], por lo que tomamos como media 4.

https://www.santander.com.mx/personas/acerca-del-banco/responsabilidad-social/assets/pdf/informe-anual/resumen-ejecutivo-del-informe-anual-integrado-2025-v1.pdf

La desviación estandar es un supuesto propio definido como 0.5.

## Abandono_riesgo	

El riesgo de abandono es inversamente proporcional a la satisfacción, el NPS y la antiguedad en meses. 

Formula: `clamp(0, 40*max(antiguedad_meses, 36))/36 + 30*nps/10 + 1.30*satisfaccion/5 + ruido, 100)`

## Segmento
Basado en gasto y antigüedad 
- Mass: Perfil inicial de tarjeta de crédito. Primera tarjeta de crédito que da el banco a nuevos clientes. Se presupone que los clientes que tienen esta tarjeta ya cuentan con un ingreso fijo demostrado por cierto tiempo.
- Select: Perfil de ingresos medios-altos. Clientes con mayor capacidad de crédito que buscan mejores beneficios. 
- Premier: Perfil de rentas altas. Clientes de muy alto valor para el banco, con alto gasto y viajes frecuentes. 

## Activacion Promos (%)
Unido a segmentos e ingreso, siguiendo estas reglas generales:
A mayor ingreso hay menor sensibilidad al precio, por lo que los clientes de ingresos altos activan menos promociones.

Dentro de cada segmento, la activación baja conforme el ingreso del cliente supera el ingreso medio de su segmento (−5 pts por cada vez que el ingreso se multiplica por e ≈ 2.7).

## Tarjeta Santander
Basado en ingreso, segmento y antigüedad, mientras más ingreso, y antigüedad mayor es el tier de tarjeta 
- Mass:
  - Pocos ingresos y alto gasto, poca antigüedad: LikeU.
  - Pocos ingresos y menor gasto, poca antinguedad: Free. Cliente que busca control de gastos.
  - Mayor ingreso dentro del segmento:  Aeroméxico Blanca o Fiesta Rewards Oro (50% / 50%).
  
- Select:
  - Mayor antigüedad dentro del segmento: Santander Platinum o Fiesta Rewards Platinum o Aeroméxico Platinum. (uniforme entre las 3).
  - Gasto menor a la media del segmento: Santander Gold.
- Premier:
  - Mayores ingresos, antigüedad y gastos: World Elite. 
  - El resto: Aeroméxico Platinum o Santander Platinum o Fiesta Rewards Platinum 

https://www.santander.com.mx/personas/credito-y-financiamiento/tarjetas-de-credito/
https://expansion.mx/finanzas-personales/2023/05/10/como-tramitar-tarjeta-credito-santander


## Uso de Tarjeta (%)
Directamente proporcional a ingreso y transacciones por mes. Estrictamente en el intervalo (0,100)


# Árbol de decisión basado en políticas

Árbol **determinístico** (no entrenado) que traduce a reglas las políticas definidas arriba para `segmento` y `tarjeta_santander`.
Son dos árboles encadenados: primero se asigna el **segmento** (gasto y antigüedad) y luego la **tarjeta** (segmento, ingreso, gasto y antigüedad).

> ⚠️ Esta sección (código, árbol y justificación) fue generada por una **IA (Claude, de Anthropic)** a partir de las políticas escritas por el equipo. Revisen los umbrales antes de usarla.

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go

# Supuestos numéricos (MXN / meses). Las políticas solo dicen "alto", "mayor", "media del segmento";
# los umbrales intra-segmento se calibraron con medianas / percentil 80 del dataset regenerado (más abajo).
U = dict(
    premier_gasto=20_000, premier_antig=36,      # Segmento Premier
    select_gasto=8_000, select_antig=12,         # Segmento Select
    mass_ingreso_alto=10_000,                    # Mass: "mayor ingreso dentro del segmento" (≈ p80 Mass)
    mass_gasto_alto=5_500,                       # Mass: "alto gasto" (LikeU) vs control de gasto (Free) (≈ mediana Mass)
    select_gasto_media=13_000,                   # Select: "media del segmento" (≈ media Select)
    select_antig_alta=120,                       # Select: "mayor antigüedad dentro del segmento" (≈ mediana Select)
    we_ingreso=40_000, we_antig=120, we_gasto=35_000,   # Premier: World Elite (≈ p80 Premier)
)

PLATINUMS = ("Santander Platinum", "Fiesta Rewards Platinum", "Aeromexico Platinum")

def nodo(pregunta, *ramas):
    """ramas: (etiqueta, condición(cliente) -> bool, hijo). La primera condición que se cumple gana."""
    return {"pregunta": pregunta, "ramas": ramas}

def hoja(*tarjetas):
    return {"hoja": tarjetas}

ARBOL_SEGMENTO = nodo(
    f"gasto ≥ {U['premier_gasto']:,}\ny antigüedad ≥ {U['premier_antig']} m",
    ("Sí", lambda c: c["gasto_mensual"] >= U["premier_gasto"] and c["antiguedad_meses"] >= U["premier_antig"], hoja("Premier")),
    ("No", lambda c: True, nodo(
        f"gasto ≥ {U['select_gasto']:,}\ny antigüedad ≥ {U['select_antig']} m",
        ("Sí", lambda c: c["gasto_mensual"] >= U["select_gasto"] and c["antiguedad_meses"] >= U["select_antig"], hoja("Select")),
        ("No", lambda c: True, hoja("Mass")),
    )),
)

ARBOL_TARJETA = nodo(
    "segmento",
    ("Mass", lambda c: c["segmento"] == "Mass", nodo(
        f"ingreso ≥ {U['mass_ingreso_alto']:,}",
        ("Sí", lambda c: c["ingreso_mensual"] >= U["mass_ingreso_alto"], hoja("Aeromexico Blanca", "Fiesta Rewards Oro")),
        ("No", lambda c: True, nodo(
            f"gasto ≥ {U['mass_gasto_alto']:,}",
            ("Sí", lambda c: c["gasto_mensual"] >= U["mass_gasto_alto"], hoja("LikeU")),
            ("No", lambda c: True, hoja("Free")),
        )),
    )),
    ("Select", lambda c: c["segmento"] == "Select", nodo(
        f"gasto < {U['select_gasto_media']:,}\n(media Select)",
        ("Sí", lambda c: c["gasto_mensual"] < U["select_gasto_media"], hoja("Santander Gold")),
        ("No", lambda c: True, nodo(
            f"antigüedad ≥ {U['select_antig_alta']} m",
            ("Sí", lambda c: c["antiguedad_meses"] >= U["select_antig_alta"], hoja(*PLATINUMS)),
            ("No", lambda c: True, hoja("Santander Gold")),  # hueco de la política: ver justificación
        )),
    )),
    ("Premier", lambda c: c["segmento"] == "Premier", nodo(
        f"ingreso ≥ {U['we_ingreso']:,}\nantigüedad ≥ {U['we_antig']} m\ngasto ≥ {U['we_gasto']:,}",
        ("Sí", lambda c: c["ingreso_mensual"] >= U["we_ingreso"] and c["antiguedad_meses"] >= U["we_antig"]
                         and c["gasto_mensual"] >= U["we_gasto"], hoja("World Elite")),
        ("No", lambda c: True, hoja(*PLATINUMS)),
    )),
)

In [ ]:
def recorrer(arbol, cliente, rng=None):
    """Devuelve (resultado, ruta). Si la hoja tiene varias tarjetas se elige uniforme, como dice la política."""
    ruta = []
    while "pregunta" in arbol:
        etiqueta, _, hijo = next(r for r in arbol["ramas"] if r[1](cliente))
        ruta.append(f"{arbol['pregunta'].replace(chr(10), ' ')} → {etiqueta}")
        arbol = hijo
    opciones = arbol["hoja"]
    rng = rng or np.random.default_rng()
    return (opciones[0] if len(opciones) == 1 else str(rng.choice(opciones))), ruta

def asignar(cliente, rng=None):
    cliente = dict(cliente)
    cliente["segmento"], _ = recorrer(ARBOL_SEGMENTO, cliente)
    tarjeta, ruta = recorrer(ARBOL_TARJETA, cliente, rng)
    return cliente["segmento"], tarjeta, " | ".join(ruta)

In [ ]:
def dibujar_arbol(arbol, titulo):
    nodos, aristas, hojas_x = [], [], [0]

    def colocar(n, prof):
        if "hoja" in n:
            x = hojas_x[0]; hojas_x[0] += 1
            nodos.append((x, -prof, "<br>".join(n["hoja"]), True))
            return x
        xs = [colocar(h, prof + 1) for _, _, h in n["ramas"]]
        x = sum(xs) / len(xs)
        nodos.append((x, -prof, n["pregunta"].replace("\n", "<br>"), False))
        for (etq, _, _), xh in zip(n["ramas"], xs):
            aristas.append((x, -prof, xh, -prof - 1, etq))
        return x

    colocar(arbol, 0)
    fig = go.Figure()
    for x0, y0, x1, y1, etq in aristas:
        fig.add_shape(type="line", x0=x0, y0=y0, x1=x1, y1=y1, line=dict(color="#a3a29c", width=1.5), layer="below")
        fig.add_annotation(x=(x0 + x1) / 2, y=(y0 + y1) / 2, text=f"<b>{etq}</b>", showarrow=False,
                           font=dict(size=11, color="#52514e"), bgcolor="#fcfcfb")
    for x, y, texto, es_hoja in nodos:
        fig.add_annotation(x=x, y=y, text=texto, showarrow=False, align="center", borderpad=6,
                           font=dict(size=11, color="#0b0b0b"),
                           bgcolor="#cde2fb" if es_hoja else "#f0efec",
                           bordercolor="#2a78d6" if es_hoja else "#52514e", borderwidth=1.5)
    profundidad = -min(y for _, y, _, _ in nodos)
    fig.update_xaxes(visible=False, range=[-0.7, hojas_x[0] - 0.3])
    fig.update_yaxes(visible=False, range=[-profundidad - 0.5, 0.5])
    fig.update_layout(title=titulo, height=170 * (profundidad + 1), plot_bgcolor="#fcfcfb",
                      paper_bgcolor="#fcfcfb", margin=dict(l=20, r=20, t=60, b=20), showlegend=False)
    return fig

dibujar_arbol(ARBOL_SEGMENTO, "Árbol 1 — Asignación de segmento (gasto y antigüedad)").show()
dibujar_arbol(ARBOL_TARJETA, "Árbol 2 — Asignación de tarjeta (segmento, ingreso, gasto y antigüedad)").show()

In [ ]:
ejemplos = pd.DataFrame([
    dict(customer_id="C000001", ingreso_mensual=6_000, gasto_mensual=3_000, antiguedad_meses=6),
    dict(customer_id="C000002", ingreso_mensual=8_000, gasto_mensual=6_000, antiguedad_meses=8),
    dict(customer_id="C000003", ingreso_mensual=30_000, gasto_mensual=7_000, antiguedad_meses=40),
    dict(customer_id="C000004", ingreso_mensual=25_000, gasto_mensual=10_000, antiguedad_meses=30),
    dict(customer_id="C000005", ingreso_mensual=35_000, gasto_mensual=16_000, antiguedad_meses=150),
    dict(customer_id="C000006", ingreso_mensual=40_000, gasto_mensual=15_000, antiguedad_meses=24),
    dict(customer_id="C000007", ingreso_mensual=60_000, gasto_mensual=30_000, antiguedad_meses=60),
    dict(customer_id="C000008", ingreso_mensual=150_000, gasto_mensual=70_000, antiguedad_meses=180),
])
rng = np.random.default_rng(42)
ejemplos[["segmento", "tarjeta_santander", "ruta"]] = [asignar(c, rng) for c in ejemplos.to_dict("records")]

esperado = ["Free", "LikeU", {"Aeromexico Blanca", "Fiesta Rewards Oro"}, "Santander Gold",
            set(PLATINUMS), "Santander Gold", set(PLATINUMS), "World Elite"]
for t, e in zip(ejemplos.tarjeta_santander, esperado):
    assert t in e if isinstance(e, set) else t == e, (t, e)
assert list(ejemplos.segmento) == ["Mass"] * 3 + ["Select"] * 3 + ["Premier"] * 2

pd.set_option("display.max_colwidth", None)
display(ejemplos)

# Generación de 10,000 clientes

Base sintética siguiendo las políticas de arriba. `segmento` y `tarjeta_santander` salen del árbol de decisión. Se guarda en `data/dataset_regenerado_10000_clientes.csv` (semilla fija → reproducible).

> ⚠️ Código generado por una **IA (Claude, de Anthropic)**. Varias fórmulas de la política se corrigieron; los cambios están en la justificación al final.

In [ ]:
N = 10_000
rng = np.random.default_rng(2026)

# Pirámide poblacional México 2026 (populationpyramid.net). Edad máxima del rango 75+ = 90 (supuesto).
RANGOS_EDAD = [(15, 24), (25, 34), (35, 44), (45, 54), (55, 64), (65, 74), (75, 90)]
PESOS_EDAD = np.array([22_244_425, 21_105_726, 18_729_814, 15_640_829, 12_087_127, 7_384_543, 4_326_566])

# shortcut: medias de ingreso supuestas, reemplazar por las cifras de DataMexico (profesionistas y técnicos, 2026-T1).
INGRESO_MEDIO = {(15, 24): 8_500, (25, 34): 13_500, (35, 44): 16_000, (45, 54): 16_500,
                 (55, 64): 15_000, (65, 74): 11_000, (75, 90): 8_000}
# La política pide σ = 500 MXN, pero eso deja a todos con casi el mismo ingreso (y sin clientes Premier).
SIGMA_LOG_INGRESO = 0.6   # dispersión en escala log (CV ≈ 66%)
SIGMA_LOG_GASTO = 0.3

# Activación de promos base por segmento: a mayor ingreso, menor sensibilidad al precio (supuesto).
PROMO_BASE = {"Mass": 45, "Select": 35, "Premier": 25}

def lognormal(media, sigma, size=None):
    """Log-normal parametrizada por su media real (no la de log)."""
    return rng.lognormal(np.log(media) - sigma**2 / 2, sigma, size)

In [ ]:
idx = rng.choice(len(RANGOS_EDAD), size=N, p=PESOS_EDAD / PESOS_EDAD.sum())
lo, hi = np.array(RANGOS_EDAD).T
edad = rng.integers(lo[idx], hi[idx] + 1)
antiguedad = rng.integers(1, (edad - 15) * 12 + 2)          # uniforme en [1, (edad-15)*12 + 1]

ingreso_medio = np.array([INGRESO_MEDIO[r] for r in RANGOS_EDAD])[idx]
ingreso = lognormal(ingreso_medio, SIGMA_LOG_INGRESO).round(-1)
gasto = np.minimum(lognormal(0.9 * ingreso, SIGMA_LOG_GASTO), 1.1 * ingreso).round(-1)  # media desplazada 10%, techo +10%

df = pd.DataFrame({
    "customer_id": [f"C{i:06d}" for i in range(1, N + 1)],
    "edad": edad, "antiguedad_meses": antiguedad, "ingreso_mensual": ingreso, "gasto_mensual": gasto,
})
df[["segmento", "tarjeta_santander", "ruta"]] = [asignar(c, rng) for c in df.to_dict("records")]

ruido = lambda sd: rng.normal(0, sd, N)
holgura = (ingreso - gasto) / ingreso                       # entre -0.1 y 1
df["pagos_puntuales_pct"] = np.clip(100 * (0.75 + 0.25 * holgura) + ruido(5), 0, 100).round(1)
df["ecommerce_pct"] = np.clip(70 * gasto / 20_000 + 30 * (100 - edad) / 100 + ruido(5), 0, 100).round(1)
df["transacciones_mes"] = np.clip(gasto / 200 + ruido(5), 0, None).round().astype(int)
df["app_logins_mes"] = np.clip(0.1 * df.ecommerce_pct + 0.2 * df.transacciones_mes + ruido(2), 0, None).round().astype(int)
df["nps"] = np.clip(rng.normal(6.92, 1.5, N), 0, 10).round().astype(int)
df["satisfaccion"] = np.clip(rng.normal(4, 0.5, N), 1, 5).round(1)
lealtad = 40 * np.minimum(antiguedad, 36) / 36 + 30 * df.nps / 10 + 30 * df.satisfaccion / 5
df["abandono_riesgo"] = (np.clip(100 - lealtad + ruido(5), 0, 100) / 100).round(3)

media_seg = df.groupby("segmento").ingreso_mensual.transform("mean")
df["activacion_promos_pct"] = np.clip(df.segmento.map(PROMO_BASE) - 5 * np.maximum(0, np.log(ingreso / media_seg))
                                      + ruido(8), 0, 100).round(1)
# (0, 100) estricto: logística sobre ingreso y transacciones estandarizados
z = lambda x: (x - x.mean()) / x.std()
uso = 100 / (1 + np.exp(-(0.8 * z(np.log(ingreso)) + 0.8 * z(df.transacciones_mes) + ruido(0.5))))
df["uso_tarjeta_pct"] = np.clip(uso.round(2), 0.01, 99.99)

COLUMNAS = ["customer_id", "edad", "segmento", "antiguedad_meses", "ingreso_mensual", "tarjeta_santander",
            "uso_tarjeta_pct", "transacciones_mes", "gasto_mensual", "ecommerce_pct", "pagos_puntuales_pct",
            "app_logins_mes", "nps", "satisfaccion", "abandono_riesgo", "activacion_promos_pct"]
clientes = df[COLUMNAS]

In [ ]:
assert clientes.customer_id.is_unique and len(clientes) == N and not clientes.isna().any().any()
assert (clientes.antiguedad_meses <= (clientes.edad - 15) * 12 + 1).all()
assert (clientes.gasto_mensual <= 1.1 * clientes.ingreso_mensual + 10).all()   # +10 por el redondeo
for c in ["pagos_puntuales_pct", "ecommerce_pct", "activacion_promos_pct"]:
    assert clientes[c].between(0, 100).all(), c
assert clientes.uso_tarjeta_pct.between(0, 100, inclusive="neither").all()
assert clientes.abandono_riesgo.between(0, 1).all() and clientes.nps.between(0, 10).all()

display(pd.crosstab(clientes.tarjeta_santander, clientes.segmento, margins=True, margins_name="Total"))
display(clientes.describe().T)
clientes.to_csv("data/dataset_regenerado_10000_clientes.csv", index=False)

## Justificación del árbol

> 🤖 **Aviso:** esta justificación y el árbol fueron elaborados por una **inteligencia artificial (Claude, de Anthropic)**. Las decisiones de negocio y los umbrales son supuestos de la IA que el equipo debe validar.

### ¿Por qué un árbol de reglas y no uno entrenado?
Las políticas de *Segmento* y *Tarjeta Santander* ya son reglas del tipo "si … entonces …". Un árbol escrito a mano las representa 1 a 1, es auditable (cada cliente trae su `ruta`) y no depende de datos, que todavía no se regeneran. Entrenar un `DecisionTreeClassifier` sobre el dataset original no serviría: el EDA mostró que sus variables son independientes (|ρ| ≈ 0.03), así que el modelo aprendería ruido.

### Árbol 1 — Segmento (política: *"Basado en gasto y antigüedad"*)
| Regla | Segmento | Por qué |
|---|---|---|
| gasto ≥ 20,000 **y** antigüedad ≥ 36 m | Premier | "Rentas altas, alto gasto, muy alto valor": pedimos gasto alto y una relación probada (3 años). |
| gasto ≥ 8,000 **y** antigüedad ≥ 12 m | Select | "Ingresos medios-altos, mayor capacidad de crédito": gasto medio y al menos 1 año de historial. |
| resto | Mass | "Primera tarjeta para nuevos clientes": todo cliente nuevo o de gasto bajo entra aquí. |

Se evalúa de arriba hacia abajo (primero Premier) para que un cliente nunca caiga en dos segmentos.

### Árbol 2 — Tarjeta (política: *"Basado en ingreso, segmento y antigüedad"*)
- **Mass**
  - Ingreso ≥ 10,000 → *Aeroméxico Blanca* o *Fiesta Rewards Oro* (50 / 50) — "Mayor ingreso dentro del segmento". Se pregunta primero porque es la única regla de Mass que no menciona "poca antigüedad".
  - Si no, gasto ≥ 5,500 → *LikeU* — "Pocos ingresos y alto gasto".
  - Si no → *Free* — "Pocos ingresos y menor gasto, cliente que busca control de gastos".
- **Select**
  - Gasto < 13,000 (media supuesta de Select) → *Santander Gold* — "Gasto menor a la media del segmento". Se pregunta primero porque es la condición más restrictiva para el beneficio.
  - Si no, antigüedad ≥ 120 m → *Santander Platinum / Fiesta Rewards Platinum / Aeroméxico Platinum* (uniforme entre las 3) — "Mayor antigüedad dentro del segmento".
  - Si no → *Santander Gold* (**hueco de la política**, ver abajo).
- **Premier**
  - Ingreso ≥ 40,000 **y** antigüedad ≥ 120 m **y** gasto ≥ 35,000 → *World Elite* — "Mayores ingresos, antigüedad y gastos" (las tres a la vez).
  - Si no → *Aeroméxico / Santander / Fiesta Rewards Platinum* — "El resto".

### Supuestos y huecos que la IA tuvo que llenar
1. **Umbrales numéricos.** Las políticas usan palabras como "alto", "mayor" o "media del segmento", sin cifras. Todos los números están en el diccionario `U`. Los de segmento son supuestos en MXN; los de dentro de cada segmento se calibraron con el dataset regenerado (mediana o percentil 80 del segmento) para que ninguna tarjeta quede vacía. Si cambian los supuestos de generación, hay que recalibrar `U`.
2. **Select con gasto ≥ media pero antigüedad baja** no está cubierto por la política. Se asignó *Santander Gold*, porque la Platinum exige antigüedad y Gold es la tarjeta de entrada del segmento.
3. **Conflicto en Select:** un cliente con antigüedad alta y gasto bajo cumple las dos reglas. Se priorizó "gasto < media → Gold", porque dar Platinum a quien gasta poco contradice el espíritu de la política.
4. **Hojas con varias tarjetas** (50/50 y "uniforme entre las 3"): el árbol elige al azar con `rng`, tal como lo dice la política; con una semilla fija el resultado es reproducible.
5. Los nombres de las tarjetas se escribieron igual que en el dataset original (`Aeromexico Platinum`, sin acento) para que se puedan cruzar.

### Validación
La celda de ejemplos tiene un cliente por cada hoja del árbol y `assert`s que comprueban el segmento y la tarjeta esperados. Si alguien cambia una regla o un umbral y rompe la política, la celda falla.

## Justificación de la generación (IA)

> 🤖 Esta sección también fue escrita por una **IA (Claude, de Anthropic)**.

Se respetó la *intención* de cada política. Donde la fórmula escrita daba columnas degeneradas (siempre 0 o siempre 100), se corrigió así:

| Columna | Política escrita | Qué se hizo | Por qué |
|---|---|---|---|
| `edad` | 7 rangos de la pirámide 2026 | Igual; el rango 75+ se acota a 75–90 | Hace falta un máximo para muestrear uniforme |
| `antiguedad_meses` | Uniforme en [1, (edad−15)·12+1] | Igual | — |
| `ingreso_mensual` | Log-normal por edad, σ = 500 MXN, medias de DataMexico | Log-normal por edad con **σ_log = 0.6**; medias **supuestas** en `INGRESO_MEDIO` | Con σ = 500 todos los de una edad ganan casi lo mismo y no habría Premier. Las medias de DataMexico no venían en el notebook: reemplácenlas |
| `gasto_mensual` | Log-normal del ingreso con media desplazada 10 %, techo +10 % | Media = 0.9 × ingreso del cliente, σ_log = 0.3, techo 1.1 × ingreso | Interpretación de "desplazada"; la nota dice que en promedio se gasta menos que el ingreso |
| `pagos_puntuales_pct` | `holgura·0.25 + ruido` en [0,100] | `100·(0.75 + 0.25·holgura) + N(0,5)` | Tal cual daba valores de 0 a 0.25 (en escala 0–100) |
| `ecommerce_pct` | `70·gasto/20000 − 30·(100−edad)` | `70·gasto/20000 + 30·(100−edad)/100 + N(0,5)` | El signo y la escala hacían que todos quedaran en 0; la política dice *inversamente* proporcional a la edad |
| `transacciones_mes` | `gasto/200 + ruido` | Igual, entero ≥ 0 | — |
| `app_logins_mes` | (fórmula copiada de ecommerce) | `0.1·ecommerce + 0.2·transacciones + N(0,2)` | La política dice proporcional a ecommerce y transacciones |
| `nps` | Normal(6.92, 1.5) | Igual, entero en [0,10] | El NPS individual es entero |
| `satisfaccion` | Normal(4, 0.5) | Igual, en [1,5] con un decimal | Igual que el dataset original |
| `abandono_riesgo` | `40·max(antig,36)/36 + 30·nps/10 + 1.30·sat/5` | `1 − (40·min(antig,36)/36 + 30·nps/10 + 30·sat/5)/100 + ruido`, en [0,1] | La política dice *inversamente* proporcional; `max`→`min` y `1.30`→`30` para que los pesos sumen 100. Escala 0–1 como en el original |
| `activacion_promos_pct` | Base por segmento, −5 pts por cada factor *e* sobre el ingreso medio del segmento | Igual; las bases (Mass 45, Select 35, Premier 25) son supuestas | La política no da las bases |
| `uso_tarjeta_pct` | Proporcional a ingreso y transacciones, en (0,100) | Logística sobre ingreso (log) y transacciones estandarizados | La logística garantiza el intervalo abierto (0,100) |

**Riesgos conocidos:** hay clientes de 15–17 años con tarjeta de crédito (así lo pide la política; en México se requiere ser mayor de edad), y `transacciones_mes` es alto (~56 en promedio) por el ticket de 200 MXN.